# 07 · Saída para o agente e o operador
**CRISP-DM:** Implantação (protótipo)

A função `consultar(recurso, t_registro, horas_antes=6)` analisa o período anterior ao registro de um incidente e devolve um **JSON explicável** para o agente de triagem ou para o operador:

- **regime esperado** no contexto (nomeado no notebook 03);
- **estado** do período (Normal / Residual);
- **características do residual**: início estimado, duração;
- **evidências**: atributos acima do limite, com valor observado × faixa normal do contexto;
- **abrangência** entre clientes;
- **lado provável** (matriz heurística);
- **motivo estatístico** e **sugestão de triagem** em linguagem natural.

In [1]:
%run ./00_funcoes.ipynb

00_funcoes carregado · raiz: /home/wsl/gh/mestrado/claude9


In [2]:
import pickle
with open(RESULTADOS / "04_modelos.pkl", "rb") as f:
    ajustados = pickle.load(f)
MET = ajustados[PROPOSTO]
slots = pd.read_csv(RESULTADOS / "03_regime_por_contexto.csv")
jan_limpo = adicionar_tipo_dia_adaptado(pd.read_parquet(RESULTADOS / f"02_janelas_{JANELA}.parquet"))
poucos = recursos_poucos_clientes(recortar(jan_limpo, TREINO))
episodios = pd.read_csv(RESULTADOS / "05_episodios.csv", parse_dates=["inicio", "fim"])
consultas_df = pd.read_csv(RESULTADOS / "06_consultas.csv", parse_dates=["t_registro"])

In [3]:
LOG = {"log_vol": "vol", "log_lat_2xx": "lat_2xx", "log_n_cli": "n_cli"}


def valor_natural(atributo, x):
    """Converte o atributo para a unidade que o operador entende."""
    if atributo in LOG:
        v = float(np.expm1(x))
        return round(v, 1) if atributo == "log_lat_2xx" else int(round(v))
    return round(float(x) * 100, 1)  # taxas e frações em %


UNIDADE = {"log_vol": "transações/h", "log_lat_2xx": "ms", "log_n_cli": "clientes"}


def fmt(atributo, v):
    return f"{v} {UNIDADE[atributo]}" if atributo in UNIDADE else f"{v}%"


def consultar(recurso, t_registro, serie, horas_antes=HORAS_ANTES, janela=JANELA, met=MET):
    """Estado do recurso no período [t_registro − horas_antes, t_registro)."""
    t_registro = pd.Timestamp(t_registro)
    passo = pd.Timedelta(janela)
    s = serie[serie["recurso"] == recurso].reset_index(drop=True)
    # sequências residuais calculadas na série inteira do recurso (o início pode anteceder o período)
    zn_all = normalizado(met, s)
    esc_all = zn_all.max(axis=1)
    res_all = (esc_all > met["tau"][recurso]).to_numpy()
    seq_all = sequencias(res_all, s["recurso"])
    m = ((s["janela"] >= t_registro - pd.Timedelta(hours=horas_antes)) & (s["janela"] + passo <= t_registro)).to_numpy()
    idx = np.flatnonzero(m)
    w = s.iloc[idx]
    med, esc_r, q05, q95 = met["modelo"].referencia(w)
    zs = met["modelo"].z(w)
    ctx = sorted(set(zip(w["tipo_dia"], w["hora"])))
    reg = slots[(slots["recurso"] == recurso)].set_index(["tipo_dia", "hora"])
    regimes = list(dict.fromkeys(reg.loc[c, "nome"] for c in ctx if c in reg.index))
    tipos = list(dict.fromkeys(NOMES_DIA[t] for t in w["tipo_dia_ad"]))
    saida = {
        "recurso": int(recurso),
        "periodo_analisado": {"inicio": str(w["janela"].min()), "fim": str(t_registro), "janelas": int(len(w)),
                              "janela": janela},
        "contexto": {"tipo_dia": tipos, "horas": faixas_horas(sorted(w["hora"].unique()))},
        "regime_esperado": regimes,
        "limiar_tau": round(float(met["tau"][recurso]), 2),
    }
    res_idx = idx[res_all[idx]]
    if len(res_idx) == 0:
        top = idx[np.argmax(esc_all.to_numpy()[idx])]
        saida.update(estado="Normal", residual=None, evidencias=[], lado_provavel=None, regra=None,
                     escore_maximo=round(float(esc_all.iloc[top]), 2))
        saida["motivo_estatistico"] = (
            f"Nas {len(w)} janelas anteriores ao registro, todos os atributos do recurso {recurso} ficaram dentro "
            f"do limite esperado para o contexto ({', '.join(tipos)}, {saida['contexto']['horas']}); "
            f"o maior escore foi {saida['escore_maximo']} para τ = {saida['limiar_tau']}.")
        saida["sugestao_triagem"] = (
            f"O recurso {recurso} operou dentro do padrão normal antes do incidente: baixa probabilidade de ser a "
            f"origem. Priorizar outros componentes do fluxo.")
        return saida
    s_ult = seq_all[res_idx[-1]]
    da_seq = np.flatnonzero(seq_all == s_ult)
    no_periodo = res_idx[seq_all[res_idx] == s_ult]
    top = no_periodo[np.argmax(esc_all.to_numpy()[no_periodo])]
    pos = int(np.flatnonzero(idx == top)[0])
    zn_top, zs_top, linha = zn_all.iloc[top], zs.iloc[pos], s.iloc[top]
    lado, regra = lado_provavel(zn_top, zs_top, linha, recurso in poucos)
    evid = []
    for a in zn_top.sort_values(ascending=False).index:
        if not zn_top[a] > 1:
            break
        evid.append({"atributo": NOMES[a], "janela": str(linha["janela"]),
                     "observado": fmt(a, valor_natural(a, linha[a])),
                     "faixa_normal_p5_p95": [fmt(a, valor_natural(a, q05.iloc[pos][a])), fmt(a, valor_natural(a, q95.iloc[pos][a]))],
                     "z": round(float(zs_top[a]), 1), "vezes_o_limite": round(float(zn_top[a]), 2)})
    abr = {"clientes_ativos": int(linha["n_cli"]), "clientes_com_4xx": int(linha["n_cli_4xx"]),
           "clientes_com_5xx": int(linha["n_cli_5xx"]), "part_maior_cliente_no_volume": f"{linha['top1_vol']:.0%}",
           "part_2_maiores_nos_4xx": f"{linha['conc_4xx']:.0%}", "part_2_maiores_nos_5xx": f"{linha['conc_5xx']:.0%}"}
    inicio = s.loc[da_seq[0], "janela"]
    saida.update(estado="Residual",
                 residual={"inicio_estimado": str(inicio), "duracao_janelas": int(len(da_seq)),
                           "janelas_residuais_no_periodo": int(len(res_idx)),
                           "persistente": bool(len(da_seq) >= 2)},
                 evidencias=evid, abrangencia=abr, lado_provavel=lado, regra=regra)
    e0 = evid[0]
    saida["motivo_estatistico"] = (
        f"Desde {inicio:%d/%m %H:%M}, o recurso {recurso} está fora do comportamento normal para o contexto "
        f"({', '.join(tipos)}). Na janela de {pd.Timestamp(e0['janela']):%H:%M}, {e0['atributo']} foi {e0['observado']} "
        f"(faixa normal {e0['faixa_normal_p5_p95'][0]} a {e0['faixa_normal_p5_p95'][1]}), "
        f"{e0['vezes_o_limite']}× o limite do recurso para esse atributo.")
    sug = {
        "provedor": f"Direcionar à equipe responsável pelo recurso {recurso} (lado provedor): {regra}; "
                    f"{abr['clientes_com_5xx'] if '5xx' in regra else abr['clientes_ativos']} clientes afetados.",
        "consumidor": f"Verificar primeiro o(s) cliente(s) consumidor(es) que concentram o desvio ({regra}); "
                      f"é provável erro de integração ou uso do lado de quem chama o recurso {recurso}.",
        "indeterminado": f"Desvio confirmado no recurso {recurso}, mas sem padrão claro de lado ({regra}); "
                         f"analisar junto com outras fontes antes de direcionar.",
    }
    saida["sugestao_triagem"] = sug[lado]
    return saida

## Estudos de caso (semente 0)
1. Pico de 5xx injetado (lado real: provedor);
2. 4xx concentrado em um cliente (lado real: consumidor);
3. Consulta de controle em período limpo;
4. Residual real no teste (sem injeção).

In [4]:
js0 = adicionar_tipo_dia_adaptado(pd.read_parquet(RESULTADOS / "05_series" / f"serie_{JANELA}_s0.parquet"))
c0 = consultas_df[(consultas_df["metodo"] == PROPOSTO) & (consultas_df["semente"] == 0)]
inc = c0[(c0["tipo"] == "incidente") & (c0["estado"] == "Residual")]


def escolhe(tipo):
    cand = inc[(inc["tipo_falha"] == tipo) & (~inc["recurso"].isin(poucos))]
    return cand.iloc[0] if len(cand) else None


casos = {}
for nome, tipo in [("provedor_5xx", "erro_5xx_provedor"), ("consumidor_4xx", "erro_4xx_consumidor")]:
    c = escolhe(tipo)
    if c is not None:
        ep = episodios[(episodios["semente"] == 0) & (episodios["episodio"] == c["episodio"])].iloc[0]
        out = consultar(c["recurso"], c["t_registro"], js0)
        out["_gabarito"] = {"tipo_falha": NOMES_TIPO[ep["tipo"]], "intensidade": ep["intensidade"],
                            "inicio_real": str(ep["inicio"]), "lado_real": ep["lado"]}
        casos[nome] = out
ctl = c0[(c0["tipo"] == "controle") & (c0["estado"] == "Normal")].iloc[0]
casos["controle_normal"] = consultar(ctl["recurso"], ctl["t_registro"], js0)
reais = pd.read_csv(RESULTADOS / "06_residual_real_teste.csv", parse_dates=["inicio"])
real = reais.sort_values("escore", ascending=False).iloc[0]
casos["residual_real"] = consultar(real["recurso"], real["inicio"] + pd.Timedelta(hours=2), js0)

for nome, out in casos.items():
    print(f"===== {nome} =====")
    print(json.dumps(out, ensure_ascii=False, indent=2, default=str))

===== provedor_5xx =====
{
  "recurso": 18139260,
  "periodo_analisado": {
    "inicio": "2026-09-03 19:00:00",
    "fim": "2026-09-04 01:00:00",
    "janelas": 6,
    "janela": "1h"
  },
  "contexto": {
    "tipo_dia": [
      "dia útil"
    ],
    "horas": "0h, 19–23h"
  },
  "regime_esperado": [
    "R1: volume baixo · dia útil · esporádico"
  ],
  "limiar_tau": 2.05,
  "estado": "Residual",
  "residual": {
    "inicio_estimado": "2026-09-03 22:00:00",
    "duracao_janelas": 4,
    "janelas_residuais_no_periodo": 3,
    "persistente": true
  },
  "evidencias": [
    {
      "atributo": "frac. clientes c/ 5xx",
      "janela": "2026-09-03 23:00:00",
      "observado": "61.5%",
      "faixa_normal_p5_p95": [
        "0.0%",
        "0.0%"
      ],
      "z": 20.5,
      "vezes_o_limite": 6.84
    },
    {
      "atributo": "taxa 5xx",
      "janela": "2026-09-03 23:00:00",
      "observado": "4.6%",
      "faixa_normal_p5_p95": [
        "0.0%",
        "0.0%"
      ],
      "z": 4.6,

In [5]:
salvar_json(casos, "07_estudos_de_caso.json")

## Plano de uso
- **Integração:** a função é exposta como ferramenta ao agente de triagem: dado o incidente (recursos envolvidos e horário de registro), o agente chama `consultar` para cada recurso e recebe o JSON acima. O campo `sugestao_triagem` apoia o encaminhamento; `motivo_estatistico` e `evidencias` sustentam a explicação ao operador.
- **Retreino:** mensal, com o último mês completo (perfil, limites por atributo e τ por validação cruzada), preservando a lista de feriados.
- **Recurso novo:** sem ao menos 2–3 semanas de histórico, o perfil por hora × tipo de dia não é confiável; a saída deve ser `indeterminado` até completar o histórico.
- **Calendário:** feriados nacionais e locais entram na lista `FERIADOS` e usam o perfil de domingo / não útil.
- **Janela:** 1 h (melhor resultado no experimento E5).
- **Limitação operacional:** manutenções não recorrentes aparecem como Residual; o operador deve cruzar com o calendário de mudanças.